# Sales Data Analysis and Business Insights (Jupyter version)
Run the cells from top to bottom (**Kernel > Restart & Run All**).

**Data:** put your CSV at `data/superstore.csv` (next to this notebook). If the file is missing, sample data is generated automatically.

## 1. Setup

In [ ]:
%pip install -q pandas numpy matplotlib seaborn streamlit

In [ ]:
%%writefile sales_analysis.py
"""
Sales Data Analysis and Business Insights
Core module: load -> clean -> feature engineering -> KPIs -> aggregations.

Works with the Kaggle "Sample Superstore" dataset (data/superstore.csv) or any CSV
with similar columns. If no file exists, a realistic synthetic dataset is generated
so the whole pipeline runs end to end.
"""
import os
import numpy as np
import pandas as pd

DATA_PATH = os.path.join(os.path.dirname(__file__), "data", "superstore.csv")

# Map common column-name variants to one standard schema
COLUMN_MAP = {
    "order id": "order_id", "order date": "order_date", "customer id": "customer_id",
    "customer name": "customer_name", "segment": "segment", "region": "region",
    "category": "category", "sub-category": "sub_category", "sub_category": "sub_category",
    "product name": "product", "product": "product", "sales": "sales",
    "quantity": "quantity", "discount": "discount", "profit": "profit",
}


# ----------------------------------------------------------------------------
# 1. Data loading
# ----------------------------------------------------------------------------
def generate_sample_data(n=5000, seed=42) -> pd.DataFrame:
    """Synthetic data shaped like a retail/superstore dataset (with dirty rows)."""
    rng = np.random.default_rng(seed)
    catalog = {
        "Technology": {"Phones": 450, "Laptops": 900, "Accessories": 60},
        "Furniture": {"Chairs": 220, "Tables": 380, "Bookcases": 260},
        "Office Supplies": {"Paper": 25, "Binders": 30, "Storage": 90},
    }
    rows = []
    dates = pd.date_range("2022-01-01", "2025-12-31")
    customers = [f"C{1000 + i}" for i in range(600)]
    for i in range(n):
        cat = rng.choice(list(catalog))
        sub = rng.choice(list(catalog[cat]))
        price = catalog[cat][sub] * rng.uniform(0.8, 1.2)
        qty = int(rng.integers(1, 8))
        disc = float(rng.choice([0, 0, 0, 0.1, 0.2, 0.3, 0.5]))
        d = rng.choice(dates)
        # seasonality: boost Nov/Dec
        if pd.Timestamp(d).month in (11, 12) and rng.random() < 0.5:
            qty += 2
        sales = price * qty * (1 - disc)
        margin = {"Technology": 0.22, "Furniture": 0.08, "Office Supplies": 0.18}[cat]
        profit = sales * (margin - disc * 0.6) + rng.normal(0, 5)
        rows.append({
            "Order ID": f"O-{100000 + i}", "Order Date": d,
            "Customer ID": rng.choice(customers),
            "Segment": rng.choice(["Consumer", "Corporate", "Home Office"], p=[.52, .30, .18]),
            "Region": rng.choice(["West", "East", "Central", "South"], p=[.32, .29, .23, .16]),
            "Category": cat, "Sub-Category": sub, "Product Name": f"{sub} Model {rng.integers(1, 6)}",
            "Sales": round(sales, 2), "Quantity": qty, "Discount": disc, "Profit": round(profit, 2),
        })
    df = pd.DataFrame(rows)
    # inject dirt: missing values and duplicates
    df.loc[rng.choice(n, 60, replace=False), "Region"] = np.nan
    df.loc[rng.choice(n, 40, replace=False), "Sales"] = np.nan
    df = pd.concat([df, df.sample(50, random_state=1)], ignore_index=True)
    return df


def load_data(path: str = DATA_PATH) -> pd.DataFrame:
    if os.path.exists(path):
        try:
            return pd.read_csv(path, encoding="utf-8")
        except UnicodeDecodeError:
            return pd.read_csv(path, encoding="latin-1")
    return generate_sample_data()


# ----------------------------------------------------------------------------
# 2. Cleaning
# ----------------------------------------------------------------------------
def clean_data(df: pd.DataFrame):
    """Returns (clean_df, report_dict)."""
    report = {"rows_raw": len(df)}
    df = df.copy()
    df.columns = [COLUMN_MAP.get(c.strip().lower(), c.strip().lower().replace(" ", "_"))
                  for c in df.columns]

    report["missing_before"] = df.isna().sum()[lambda s: s > 0].to_dict()

    # duplicates
    report["duplicates_removed"] = int(df.duplicated().sum())
    df = df.drop_duplicates()

    # types
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
    for col in ("sales", "quantity", "discount", "profit"):
        if col in df:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # missing values: drop rows missing critical fields, impute the rest
    critical = [c for c in ("order_date", "sales") if c in df]
    report["rows_dropped_critical_na"] = int(df[critical].isna().any(axis=1).sum())
    df = df.dropna(subset=critical)
    for col in df.select_dtypes(include="object"):
        df[col] = df[col].fillna("Unknown").str.strip()
    for col in ("quantity", "discount", "profit"):
        if col in df:
            df[col] = df[col].fillna(df[col].median())

    # remove invalid rows
    df = df[(df["sales"] > 0)]
    if "quantity" in df:
        df = df[df["quantity"] > 0]

    # outliers flagged (IQR) but kept - they are often genuine big orders
    q1, q3 = df["sales"].quantile([.25, .75])
    iqr = q3 - q1
    df["is_outlier"] = (df["sales"] > q3 + 1.5 * iqr) | (df["sales"] < q1 - 1.5 * iqr)
    report["outliers_flagged"] = int(df["is_outlier"].sum())

    # feature engineering
    df["year"] = df["order_date"].dt.year
    df["month"] = df["order_date"].dt.month
    df["month_name"] = df["order_date"].dt.strftime("%b")
    df["quarter"] = df["order_date"].dt.quarter
    df["weekday"] = df["order_date"].dt.day_name()
    df["year_month"] = df["order_date"].dt.to_period("M").dt.to_timestamp()
    if "profit" in df:
        df["profit_margin"] = df["profit"] / df["sales"]

    report["rows_clean"] = len(df)
    return df.reset_index(drop=True), report


# ----------------------------------------------------------------------------
# 3. KPIs and aggregations
# ----------------------------------------------------------------------------
def compute_kpis(df: pd.DataFrame) -> dict:
    kpis = {
        "total_sales": df["sales"].sum(),
        "total_orders": df["order_id"].nunique(),
        "total_customers": df["customer_id"].nunique(),
        "avg_order_value": df.groupby("order_id")["sales"].sum().mean(),
        "total_quantity": int(df["quantity"].sum()),
    }
    if "profit" in df:
        kpis["total_profit"] = df["profit"].sum()
        kpis["profit_margin_pct"] = 100 * df["profit"].sum() / df["sales"].sum()
    # year over year growth
    yearly = df.groupby("year")["sales"].sum().sort_index()
    kpis["yoy_growth_pct"] = float(yearly.pct_change().iloc[-1] * 100) if len(yearly) > 1 else np.nan
    return kpis


def group_summary(df: pd.DataFrame, by) -> pd.DataFrame:
    agg = {"sales": "sum", "quantity": "sum", "order_id": "nunique"}
    if "profit" in df:
        agg["profit"] = "sum"
    out = df.groupby(by).agg(agg).rename(columns={"order_id": "orders"})
    if "profit" in out:
        out["margin_pct"] = 100 * out["profit"] / out["sales"]
    return out.sort_values("sales", ascending=False)


def monthly_sales(df: pd.DataFrame) -> pd.DataFrame:
    m = df.groupby("year_month")["sales"].sum().to_frame()
    m["rolling_3m"] = m["sales"].rolling(3).mean()
    m["mom_growth_pct"] = m["sales"].pct_change() * 100
    return m


def rfm_segmentation(df: pd.DataFrame) -> pd.DataFrame:
    """Recency / Frequency / Monetary customer segmentation."""
    snapshot = df["order_date"].max() + pd.Timedelta(days=1)
    rfm = df.groupby("customer_id").agg(
        recency=("order_date", lambda s: (snapshot - s.max()).days),
        frequency=("order_id", "nunique"),
        monetary=("sales", "sum"),
    )
    rfm["r"] = pd.qcut(rfm["recency"], 4, labels=[4, 3, 2, 1]).astype(int)
    rfm["f"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
    rfm["m"] = pd.qcut(rfm["monetary"], 4, labels=[1, 2, 3, 4]).astype(int)
    rfm["score"] = rfm["r"] + rfm["f"] + rfm["m"]

    def label(s):
        if s >= 10: return "Champions"
        if s >= 8: return "Loyal"
        if s >= 6: return "Potential"
        if s >= 4: return "At Risk"
        return "Lost"
    rfm["segment"] = rfm["score"].apply(label)
    return rfm


def generate_insights(df: pd.DataFrame) -> list:
    """Plain-English business insights computed from the data."""
    ins = []
    cat = group_summary(df, "category")
    ins.append(f"Top category by sales: {cat.index[0]} (${cat['sales'].iloc[0]:,.0f}, "
               f"{100 * cat['sales'].iloc[0] / cat['sales'].sum():.1f}% of revenue).")
    if "profit" in df:
        sub = group_summary(df, "sub_category")
        worst = sub["margin_pct"].idxmin()
        ins.append(f"Lowest-margin sub-category: {worst} ({sub.loc[worst, 'margin_pct']:.1f}% margin) "
                   f"- review pricing and discounts.")
        hi = df[df["discount"] >= 0.3]
        lo = df[df["discount"] < 0.3]
        if len(hi) and len(lo):
            ins.append(f"Orders with >=30% discount earn {100 * hi['profit'].sum() / hi['sales'].sum():.1f}% "
                       f"margin vs {100 * lo['profit'].sum() / lo['sales'].sum():.1f}% otherwise.")
    reg = group_summary(df[df["region"] != "Unknown"], "region")
    ins.append(f"Best region: {reg.index[0]}; weakest: {reg.index[-1]}.")
    m = df.groupby("month_name")["sales"].sum()
    ins.append(f"Peak month overall: {m.idxmax()}; slowest: {m.idxmin()} - plan inventory and promotions accordingly.")
    top10 = df.groupby("customer_id")["sales"].sum().nlargest(max(1, int(df['customer_id'].nunique() * .1))).sum()
    ins.append(f"Top 10% of customers generate {100 * top10 / df['sales'].sum():.1f}% of sales.")
    return ins


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import sales_analysis as sa

sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.float_format', '{:,.2f}'.format)

## 2. Load data
Create a folder named `data` next to this notebook and save your CSV there as `superstore.csv`. Otherwise sample data is used.

In [ ]:
import os
os.makedirs('data', exist_ok=True)
print('Data file found:', os.path.exists('data/superstore.csv'))

raw = sa.load_data()
print('Raw shape:', raw.shape)
raw.head()

## 3. Clean data

In [ ]:
df, report = sa.clean_data(raw)
for k, v in report.items():
    print(f'{k}: {v}')
df.head()

## 4. Descriptive statistics and KPIs

In [ ]:
display(df[['sales','quantity','discount','profit']].describe().round(2))
kpis = sa.compute_kpis(df)
pd.Series(kpis).round(2).to_frame('value')

## 5. Sales trend and seasonality

In [ ]:
m = sa.monthly_sales(df)
plt.figure(figsize=(11,4.5))
plt.plot(m.index, m['sales'], marker='o', ms=3, label='Monthly sales')
plt.plot(m.index, m['rolling_3m'], lw=2.5, label='3-month average')
plt.title('Monthly Sales Trend'); plt.ylabel('Sales ($)'); plt.legend(); plt.show()

pivot = df.pivot_table(index='year', columns='month', values='sales', aggfunc='sum')
plt.figure(figsize=(10,3.5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGnBu')
plt.title('Sales Heatmap (Year x Month)'); plt.show()

## 6. Category, region, products

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4.5))
cat = sa.group_summary(df, 'category')
sns.barplot(x=cat.index, y=cat['sales'], ax=ax[0]); ax[0].set_title('Sales by Category')
reg = sa.group_summary(df, 'region')
sns.barplot(x=reg.index, y=reg['sales'], ax=ax[1]); ax[1].set_title('Sales by Region')
plt.tight_layout(); plt.show()

top = sa.group_summary(df, 'product').head(10)
plt.figure(figsize=(9,5))
sns.barplot(y=top.index, x=top['sales'], orient='h'); plt.title('Top 10 Products by Sales'); plt.show()

sa.group_summary(df, ['category','sub_category']).round(2)

## 7. Profitability and discount impact

In [ ]:
sub = sa.group_summary(df, 'sub_category').sort_values('margin_pct')
plt.figure(figsize=(9,5))
sns.barplot(y=sub.index, x=sub['margin_pct'], orient='h')
plt.axvline(0, color='k', lw=1); plt.title('Profit Margin % by Sub-Category'); plt.show()

plt.figure(figsize=(7,5))
sns.boxplot(data=df, x='discount', y='profit_margin')
plt.title('Profit Margin by Discount Level'); plt.show()

plt.figure(figsize=(5.5,4.5))
sns.heatmap(df[['sales','quantity','discount','profit']].corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix'); plt.show()

## 8. Customer segmentation (RFM)

In [ ]:
rfm = sa.rfm_segmentation(df)
seg = rfm['segment'].value_counts()
plt.figure(figsize=(7,4))
sns.barplot(x=seg.index, y=seg.values); plt.title('Customer Segments (RFM)'); plt.ylabel('Customers'); plt.show()
rfm.sort_values('monetary', ascending=False).head(10)

## 9. Business insights

In [ ]:
for i in sa.generate_insights(df):
    print('*', i)

## 10. Save results

In [ ]:
df.to_csv('cleaned_sales.csv', index=False)
print('Saved cleaned_sales.csv in', os.getcwd())

## 11. (Optional) Streamlit dashboard
Streamlit runs in its own process, so launch it from a terminal (not from this notebook):
```
streamlit run app.py
```
The next cell writes `app.py` into this folder.

In [ ]:
%%writefile app.py
"""Interactive dashboard.  Run: streamlit run app.py"""
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt
import seaborn as sns
import sales_analysis as sa

st.set_page_config(page_title="Sales Dashboard", page_icon="📊", layout="wide")
sns.set_theme(style="whitegrid")


@st.cache_data
def get_data(uploaded=None):
    raw = pd.read_csv(uploaded, encoding="latin-1") if uploaded else sa.load_data()
    return sa.clean_data(raw)


st.title("📊 Sales Data Analysis and Business Insights")

# ---------------- Sidebar ----------------
uploaded = st.sidebar.file_uploader("Upload your sales CSV (optional)", type="csv")
df_all, report = get_data(uploaded)

st.sidebar.header("Filters")
years = sorted(df_all["year"].unique())
sel_years = st.sidebar.multiselect("Year", years, default=years)
sel_regions = st.sidebar.multiselect("Region", sorted(df_all["region"].unique()),
                                     default=sorted(df_all["region"].unique()))
sel_cats = st.sidebar.multiselect("Category", sorted(df_all["category"].unique()),
                                  default=sorted(df_all["category"].unique()))

df = df_all[df_all["year"].isin(sel_years) & df_all["region"].isin(sel_regions)
            & df_all["category"].isin(sel_cats)]
if df.empty:
    st.warning("No data for the selected filters.")
    st.stop()

# ---------------- KPI cards ----------------
k = sa.compute_kpis(df)
c1, c2, c3, c4, c5 = st.columns(5)
c1.metric("Total Sales", f"${k['total_sales']:,.0f}")
c2.metric("Total Profit", f"${k.get('total_profit', 0):,.0f}")
c3.metric("Profit Margin", f"{k.get('profit_margin_pct', 0):.1f}%")
c4.metric("Orders", f"{k['total_orders']:,}")
c5.metric("Avg Order Value", f"${k['avg_order_value']:,.0f}")

tab1, tab2, tab3, tab4, tab5 = st.tabs(
    ["📈 Trends", "🛒 Products", "🌍 Regions", "👥 Customers", "💡 Insights & Data"])

# ---------------- Trends ----------------
with tab1:
    m = sa.monthly_sales(df)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(m.index, m["sales"], marker="o", ms=3, label="Monthly")
    ax.plot(m.index, m["rolling_3m"], lw=2.5, label="3-month avg")
    ax.set_title("Monthly Sales"); ax.legend()
    st.pyplot(fig)
    col1, col2 = st.columns(2)
    with col1:
        pivot = df.pivot_table(index="year", columns="month", values="sales", aggfunc="sum")
        fig, ax = plt.subplots(figsize=(6, 3))
        sns.heatmap(pivot, cmap="YlGnBu", ax=ax); ax.set_title("Seasonality heatmap")
        st.pyplot(fig)
    with col2:
        wd = df.groupby("weekday")["sales"].sum().reindex(
            ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
        fig, ax = plt.subplots(figsize=(6, 3))
        sns.barplot(x=wd.index, y=wd.values, ax=ax)
        ax.set_title("Sales by weekday"); plt.setp(ax.get_xticklabels(), rotation=45)
        st.pyplot(fig)

# ---------------- Products ----------------
with tab2:
    col1, col2 = st.columns(2)
    with col1:
        top = sa.group_summary(df, "product").head(10)
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.barplot(y=top.index, x=top["sales"], ax=ax, orient="h"); ax.set_title("Top 10 products")
        st.pyplot(fig)
    with col2:
        sub = sa.group_summary(df, "sub_category").sort_values("margin_pct")
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.barplot(y=sub.index, x=sub["margin_pct"], ax=ax, orient="h")
        ax.set_title("Margin % by sub-category")
        st.pyplot(fig)
    st.dataframe(sa.group_summary(df, ["category", "sub_category"]).round(2))

# ---------------- Regions ----------------
with tab3:
    reg = sa.group_summary(df, "region")
    col1, col2 = st.columns(2)
    with col1:
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.pie(reg["sales"], labels=reg.index, autopct="%1.1f%%", startangle=90)
        ax.set_title("Share of sales by region")
        st.pyplot(fig)
    with col2:
        st.dataframe(reg.round(2))
        seg = sa.group_summary(df, "segment")
        st.dataframe(seg.round(2))

# ---------------- Customers ----------------
with tab4:
    rfm = sa.rfm_segmentation(df)
    col1, col2 = st.columns(2)
    with col1:
        s = rfm["segment"].value_counts()
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.barplot(x=s.index, y=s.values, ax=ax); ax.set_title("RFM customer segments")
        st.pyplot(fig)
    with col2:
        st.subheader("Top 10 customers")
        st.dataframe(rfm.sort_values("monetary", ascending=False).head(10)[
            ["recency", "frequency", "monetary", "segment"]].round(0))

# ---------------- Insights ----------------
with tab5:
    st.subheader("Key business insights")
    for i in sa.generate_insights(df):
        st.markdown(f"- {i}")
    with st.expander("Data cleaning report"):
        st.json({k: (v if not isinstance(v, dict) else v) for k, v in report.items()})
    st.download_button("Download filtered data (CSV)", df.to_csv(index=False), "filtered_sales.csv")
